# Composite leakage / selection experiment - on YOUR data

Runs the same check as the synthetic version, but on a real labelled table (an `eval.csv`). It answers,
**for your data**: how much does the in-sample composite estimate overstate the honest CV estimate, does
PR-AUC weighting help, does the composite beat the best single metric, and how stable is best-per-family
selection?

It reuses the real pipeline pieces (`get_numeric_metrics`, YAML directions/families, `WINDOW_FAMILIES`),
so running it top-to-bottom is also a quick integration test that those imports work on your machine.
Metrics are direction-aligned (higher = better) and NaN is handled per metric / per row.

## Setup
Point `PROJECT_ROOT` at the folder that contains `config/`, `analysis/`, `preprocessing/` (i.e. `project_current`).

In [1]:
import sys, os
from pathlib import Path

# notebooks/ -> project_current ; override with the PROJECT_ROOT env var if you launch elsewhere
PROJECT_ROOT = Path(os.environ.get('PROJECT_ROOT', Path.cwd().parent))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from collections import Counter
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit

from config.paths import METRIC_YAML, RAW_DATA_DIR
from config.datasets import cfg
from config.analysis import WINDOW_FAMILIES
from analysis.distributions import get_numeric_metrics          # central feature selector (drops meta/experimental cols)
from preprocessing.align import load_metric_directions, get_direction
from analysis.evaluation.metrics import load_metric_families, get_family

directions = load_metric_directions(METRIC_YAML)
families   = load_metric_families(METRIC_YAML)

## Load your data
Defaults to the current `DATASET`'s `eval.csv`. Change `EVAL_CSV` to any labelled table you want to test
(it must have a `binder` 0/1 column and the metric columns).

In [2]:
EVAL_CSV = RAW_DATA_DIR / cfg.name / 'eval.csv'
# EVAL_CSV = Path('/scicore/home/schwede/barta0000/project/data/raw/current/eval.csv')  # or set explicitly

def load_data(csv_path, drop_window=True):
    """Return (df, y, cols, fam): labelled rows, binary y, metric feature columns, and metric->family.
    Features come from get_numeric_metrics (drops sample/source/type/binder/binder_type/dataset/iteration/
    KD[M]/EC50[M]); window families (net_charge, surface_hydrophobicity) are dropped from monotonic scoring."""
    df = pd.read_csv(csv_path)
    y = pd.to_numeric(df['binder'], errors='coerce')
    keep = y.isin([0, 1])
    df = df[keep].reset_index(drop=True)
    y = y[keep].astype(int).to_numpy()
    cols = get_numeric_metrics(df)
    if drop_window:
        cols = [c for c in cols if get_family(c, families) not in WINDOW_FAMILIES]
    fam = {c: (get_family(c, families) or c) for c in cols}
    return df, y, cols, fam

df, y, cols, fam = load_data(EVAL_CSV)
assert np.bincount(y).min() >= 2, 'need >=2 samples in each class'
nan_frac = pd.Series({c: float(pd.to_numeric(df[c], errors='coerce').isna().mean()) for c in cols})
print(f'file      : {EVAL_CSV}')
print(f'rows      : {len(df)}   classes (0/1): {np.bincount(y).tolist()}')
print(f'metrics   : {len(cols)}   families: {len(set(fam.values()))}')
print(f'NaN-heavy metrics (top 5):')
print(nan_frac.sort_values(ascending=False).head().round(3).to_string())

file      : /scicore/home/schwede/barta0000/project/data/raw/rf/eval.csv
rows      : 129   classes (0/1): [65, 64]
metrics   : 49   families: 14
NaN-heavy metrics (top 5):
af3_pae       0.0
af3_plddt     0.0
af3_ipae      0.0
af3_iplddt    0.0
af3_ipsae     0.0


## Helpers (direction-aligned, NaN-safe)

In [3]:
def aligned(frame, columns):
    """Direction-aligned metric matrix (higher = better) as a DataFrame."""
    signs = np.array([get_direction(c, directions) for c in columns], float)
    return pd.DataFrame(frame[columns].to_numpy(float) * signs, index=frame.index, columns=list(columns))

def prauc_per_metric(A, yy):
    """PR-AUC per aligned metric on its non-NaN rows (skips metrics that can't be scored)."""
    out = {}
    for c in A.columns:
        v = A[c].to_numpy(); m = ~np.isnan(v)
        if m.sum() >= 2 and len(np.unique(yy[m])) == 2:
            out[c] = average_precision_score(yy[m], v[m])
    return out

def best_per_family(prauc, fammap):
    best = {}
    for c, p in prauc.items():
        f = fammap[c]
        if f not in best or p > prauc[best[f]]:
            best[f] = c
    return list(best.values())

def zfit(A, sel, idx):
    """Per-column mean/std of the selected metrics on the train rows only (NaN-aware)."""
    sub = A.iloc[idx][sel].to_numpy()
    mu = np.nanmean(sub, 0); sd = np.nanstd(sub, 0); sd = np.where(sd > 0, sd, 1.0)
    return mu, sd

def composite(A, sel, mu, sd, weights):
    """Weighted mean of standardized selected metrics; NaN skipped per row (weights renormalized)."""
    Z = (A[sel].to_numpy() - mu) / sd
    w = np.asarray(weights, float)
    if w.sum() == 0: w = np.ones(len(sel))
    present = ~np.isnan(Z)
    denom = (present * w).sum(1); num = np.nansum(np.where(present, Z, 0.0) * w, 1)
    return np.divide(num, denom, out=np.full(len(A), np.nan), where=denom > 0)

def auroc_valid(yy, s):
    m = ~np.isnan(s)
    return roc_auc_score(yy[m], s[m]) if m.sum() and len(np.unique(yy[m])) == 2 else np.nan

## Run the experiment
- **in-sample_weighted (leaky)**: select best-per-family + PR-AUC-weight + score, all on the full set.
- **holdout_weighted / cv_weighted**: select + weight on train folds, score held-out (honest).
- **cv_consensus**: equal-weight (no label-derived weights), out-of-fold.
- **best_single_holdout**: the strongest single metric, held-out - the baseline to beat.
Over `seeds` random re-splits of *your* data (data is fixed; the splits vary).

In [4]:
A = aligned(df, cols)   # direction applied once

def run(seeds=25, k=5):
    # in-sample (leaky): everything on the full set
    pr_full = prauc_per_metric(A, y); selF = best_per_family(pr_full, fam)
    muF, sdF = zfit(A, selF, np.arange(len(df)))
    insample = auroc_valid(y, composite(A, selF, muF, sdF, [pr_full[c] for c in selF]))

    res = {'holdout_weighted': [], 'cv_weighted': [], 'cv_consensus': [], 'best_single_holdout': []}
    sel_sets = []
    kk = int(min(k, np.bincount(y).min()))
    for s in range(seeds):
        (tr, te), = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=s).split(A, y)
        prT = prauc_per_metric(A.iloc[tr], y[tr])
        if not prT:
            continue
        selB = best_per_family(prT, fam); sel_sets.append(tuple(sorted(selB)))
        mu, sd = zfit(A, selB, tr)
        res['holdout_weighted'].append(auroc_valid(y[te], composite(A.iloc[te], selB, mu, sd, [prT[c] for c in selB])))
        best_c = max(prT, key=prT.get)
        vB = A.iloc[te][best_c].to_numpy(); m = ~np.isnan(vB)
        res['best_single_holdout'].append(roc_auc_score(y[te][m], vB[m]) if len(np.unique(y[te][m])) == 2 else np.nan)
        oof_w = np.full(len(df), np.nan); oof_c = np.full(len(df), np.nan)
        for trk, tek in StratifiedKFold(kk, shuffle=True, random_state=s).split(A, y):
            prk = prauc_per_metric(A.iloc[trk], y[trk])
            if not prk:
                continue
            sk = best_per_family(prk, fam); mu, sd = zfit(A, sk, trk)
            oof_w[tek] = composite(A.iloc[tek], sk, mu, sd, [prk[c] for c in sk])
            oof_c[tek] = composite(A.iloc[tek], sk, mu, sd, np.ones(len(sk)))
        res['cv_weighted'].append(auroc_valid(y, oof_w))
        res['cv_consensus'].append(auroc_valid(y, oof_c))
    stab = Counter(sel_sets).most_common(1)[0][1] / max(1, len(sel_sets))
    stats = {kk2: (np.nanmean(v), np.nanstd(v)) for kk2, v in res.items() if len(v)}
    return insample, stats, stab

insample, stats, stab = run(seeds=25, k=5)

In [5]:
rows = [('insample_weighted (leaky)', insample, 0.0)]
for kk2 in ['holdout_weighted', 'cv_weighted', 'cv_consensus', 'best_single_holdout']:
    m, sd = stats.get(kk2, (np.nan, np.nan)); rows.append((kk2, m, sd))
summary = pd.DataFrame(rows, columns=['approach', 'auroc_mean', 'auroc_sd']).set_index('approach')
summary['AUROC'] = [f'{m:.3f} ± {s:.3f}' for m, s in zip(summary.auroc_mean, summary.auroc_sd)]
print(summary[['AUROC']].to_string())
print()
if 'cv_weighted' in stats:
    print(f"optimism gap (in-sample - CV weighted)          : {insample - stats['cv_weighted'][0]:+.3f}")
    print(f"weighting benefit (CV weighted - CV consensus)  : {stats['cv_weighted'][0] - stats['cv_consensus'][0]:+.3f}")
    print(f"composite (CV consensus) vs best single (holdout): {stats['cv_consensus'][0] - stats['best_single_holdout'][0]:+.3f}")
print(f"best-per-family selection identical across seeds : {stab*100:.0f}%")

                                   AUROC
approach                                
insample_weighted (leaky)  0.674 ± 0.000
holdout_weighted           0.642 ± 0.058
cv_weighted                0.667 ± 0.004
cv_consensus               0.659 ± 0.005
best_single_holdout        0.626 ± 0.060

optimism gap (in-sample - CV weighted)          : +0.007
weighting benefit (CV weighted - CV consensus)  : +0.008
composite (CV consensus) vs best single (holdout): +0.033
best-per-family selection identical across seeds : 16%


## How to read it (for your data)
- **optimism gap > ~0.02-0.03** confirms the old in-sample composite numbers were inflated - trust the CV row.
- **low selection stability** (well under 100%) means best-per-family picks are noisy on your data - prefer a
  fixed/domain metric set or stability selection (as `analysis/composite/cv.py` does).
- **composite (CV consensus) vs best single**: if this is <= 0, the composite is not beating the best single
  metric out of sample, and a single metric may be the honest recommendation.
- Re-run with `run(seeds=..., k=...)`, or point `EVAL_CSV` at a per-dataset `eval.csv` to check one system at a time.